In [7]:
if 'spark' in globals():
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 12:54:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-10-27-Oct"



KAFKA_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-30-Sep"



KAFKA_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [4]:
from pyspark.sql.types import *

print("=" * 70)
print("KAFKA BRONZE — EDA")
print("=" * 70)



df_kafka = spark.read.parquet(KAFKA_BRONZE_PATH)

print("Kafka Bronze loaded successfully.")

KAFKA BRONZE — EDA


26/10/28 12:55:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Kafka Bronze loaded successfully.


In [5]:
print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

record_count = df_kafka.count()

print(f"Total records: {record_count:,}")
print(f"Total columns: {len(df_kafka.columns)}")

print("\nColumns:")
for column_name in df_kafka.columns:
    print(f"- {column_name}")

1. DATASET OVERVIEW


[Stage 1:>                                                          (0 + 1) / 1]

Total records: 561
Total columns: 7

Columns:
- event_id
- event_timestamp
- user_id
- video_id
- interaction_type
- watch_time_sec
- device_type


In [4]:
print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

record_count = df_kafka.count()

print(f"Total records: {record_count:,}")
print(f"Total columns: {len(df_kafka.columns)}")

print("\nColumns:")
for column_name in df_kafka.columns:
    print(f"- {column_name}")

1. DATASET OVERVIEW


[Stage 1:>                                                          (0 + 1) / 1]

Total records: 14,728
Total columns: 7

Columns:
- event_id
- event_timestamp
- user_id
- video_id
- interaction_type
- watch_time_sec
- device_type


In [5]:
df_kafka.select("video_id").distinct().count()

5041

In [6]:
print("=" * 70)
print("2. SCHEMA")
print("=" * 70)

df_kafka.printSchema()

2. SCHEMA
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- interaction_type: string (nullable = true)
 |-- watch_time_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)



In [6]:
print("=" * 70)
print("3. SAMPLE DATA")
print("=" * 70)

df_kafka.show(5, truncate=False, vertical=True)

3. SAMPLE DATA


-RECORD 0------------------------------------------------
 event_id         | 01a17e5c-aeca-7ccd-8105-56e7029a94a3 
 event_timestamp  | 2026-01-17 00:47:47+00:00            
 user_id          | 01a17e5c-9c12-7fb9-ac4d-750eb39a2b34 
 video_id         | 01a17e5c-9b31-7e81-96d2-8b76ac75b2d6 
 interaction_type | play                                 
 watch_time_sec   | 1                                    
 device_type      | mobile                               
-RECORD 1------------------------------------------------
 event_id         | 01a17e5c-af12-7fae-a3a3-e02fc7602622 
 event_timestamp  | 2026-01-17 00:47:47+00:00            
 user_id          | 01a17e5c-9d0a-7252-a81d-9f993afe390f 
 video_id         | 01a17e5c-9aa9-7dbc-969f-d0366b1080cb 
 interaction_type | play                                 
 watch_time_sec   | 115                                  
 device_type      | mobile                               
-RECORD 2------------------------------------------------
 event_id     

In [7]:
print("=" * 70)
print("3. SAMPLE DATA")
print("=" * 70)

df_kafka.show(5, truncate=False, vertical=True)

3. SAMPLE DATA
-RECORD 0------------------------------------------------
 event_id         | 01a0f05b-d18d-7129-84ba-f8e558db1014 
 event_timestamp  | 2026-01-01 07:46:59+00:00            
 user_id          | 01a0f05b-06c3-7f4c-a2d3-fb1bb56a0757 
 video_id         | 01a0f05a-eba1-7433-95de-0bbbdd2c13f1 
 interaction_type | like                                 
 watch_time_sec   | NULL                                 
 device_type      | mobile                               
-RECORD 1------------------------------------------------
 event_id         | 01a0f05b-d227-7a86-90fd-4293011c92ec 
 event_timestamp  | 2026-01-02 01:28:00+00:00            
 user_id          | 01a0f05b-072b-7d36-8096-96f815e0433e 
 video_id         | 01a0f05b-0610-7abd-a0b1-1063e8b39b27 
 interaction_type | like                                 
 watch_time_sec   | 30                                   
 device_type      | web                                  
-RECORD 2------------------------------------------------

In [8]:
print("=" * 70)
print("4. DATA TYPES")
print("=" * 70)

for field in df_kafka.schema.fields:
    print(f"{field.name}: {field.dataType}")

4. DATA TYPES
event_id: StringType()
event_timestamp: StringType()
user_id: StringType()
video_id: StringType()
interaction_type: StringType()
watch_time_sec: IntegerType()
device_type: StringType()


In [9]:
print("=" * 70)
print("5. NULL / MISSING VALUES")
print("=" * 70)

null_counts = df_kafka.select([
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)
    for column_name in df_kafka.columns
])

null_counts.show()

5. NULL / MISSING VALUES
+--------+---------------+-------+--------+----------------+--------------+-----------+
|event_id|event_timestamp|user_id|video_id|interaction_type|watch_time_sec|device_type|
+--------+---------------+-------+--------+----------------+--------------+-----------+
|       0|              0|      0|       0|               0|         11909|          0|
+--------+---------------+-------+--------+----------------+--------------+-----------+



In [10]:
print("=" * 70)
print("6. EVENT ID UNIQUENESS")
print("=" * 70)

duplicate_event_ids = (
    df_kafka
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_event_count = duplicate_event_ids.count()

print(f"Duplicate event_id groups: {duplicate_event_count}")

6. EVENT ID UNIQUENESS
Duplicate event_id groups: 0


In [11]:
null_event_ids = (
    df_kafka
    .filter(F.col("event_id").isNull())
    .count()
)

print(f"Null event_id: {null_event_ids}")

Null event_id: 0


In [12]:
print("=" * 70)
print("7. USER / VIDEO RELATIONSHIPS")
print("=" * 70)

distinct_users = df_kafka.select("user_id").distinct().count()
distinct_videos = df_kafka.select("video_id").distinct().count()
distinct_pairs = (
    df_kafka
    .select("user_id", "video_id")
    .distinct()
    .count()
)

print(f"Distinct users: {distinct_users:,}")
print(f"Distinct videos: {distinct_videos:,}")
print(f"Distinct user-video pairs: {distinct_pairs:,}")

7. USER / VIDEO RELATIONSHIPS
Distinct users: 6,658
Distinct videos: 5,041
Distinct user-video pairs: 14,714


In [13]:
print("Top users by event count:")

(
    df_kafka
    .groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
    .show(10)
)

Top users by event count:
+--------------------+-----+
|             user_id|count|
+--------------------+-----+
|01a0f05b-08e5-731...|    3|
|01a0f05b-095f-7d1...|    3|
|01a0f05b-06b9-7ef...|    3|
|01a0f05b-07b9-725...|    3|
|01a0f05b-09eb-7cd...|    3|
|01a0f05b-073f-73a...|    3|
|01a0f05b-073d-72b...|    3|
|01a0f05b-092b-787...|    3|
|01a0f05b-0747-7de...|    3|
|01a0f05b-094f-75c...|    3|
+--------------------+-----+
only showing top 10 rows



In [14]:
print("=" * 70)
print("8. INTERACTION TYPE DISTRIBUTION")
print("=" * 70)

(
    df_kafka
    .groupBy("interaction_type")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

8. INTERACTION TYPE DISTRIBUTION
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|like            |3784 |
|complete        |3726 |
|pause           |3625 |
|play            |3592 |
|share           |1    |
+----------------+-----+



In [15]:
print("=" * 70)
print("9. WATCH TIME ANALYSIS")
print("=" * 70)

df_kafka.select(
    F.count("watch_time_sec").alias("non_null_watch_time"),
    F.sum(
        F.when(F.col("watch_time_sec").isNull(), 1).otherwise(0)
    ).alias("null_watch_time"),
    F.min("watch_time_sec").alias("min_watch_time"),
    F.max("watch_time_sec").alias("max_watch_time"),
    F.avg("watch_time_sec").alias("avg_watch_time"),
).show()

9. WATCH TIME ANALYSIS
+-------------------+---------------+--------------+--------------+------------------+
|non_null_watch_time|null_watch_time|min_watch_time|max_watch_time|    avg_watch_time|
+-------------------+---------------+--------------+--------------+------------------+
|               2819|          11909|             0|         20000|1904.3763746009224|
+-------------------+---------------+--------------+--------------+------------------+



In [16]:
print("Watch time <= 0:")

df_kafka.filter(
    F.col("watch_time_sec").isNotNull()
    & (F.col("watch_time_sec") <= 0)
).count()

Watch time <= 0:


1

In [17]:
print("Watch time > 120:")

df_kafka.filter(
    F.col("watch_time_sec").isNotNull()
    & (F.col("watch_time_sec") > 120)
).count()

Watch time > 120:


1427

In [18]:
print("=" * 70)
print("10. INTERACTION TYPE / WATCH TIME")
print("=" * 70)

(
    df_kafka
    .groupBy("interaction_type")
    .agg(
        F.count("*").alias("events"),
        F.sum(
            F.when(F.col("watch_time_sec").isNull(), 1)
             .otherwise(0)
        ).alias("null_watch_time"),
        F.sum(
            F.when(F.col("watch_time_sec").isNotNull(), 1)
             .otherwise(0)
        ).alias("non_null_watch_time")
    )
    .orderBy("interaction_type")
    .show()
)

10. INTERACTION TYPE / WATCH TIME
+----------------+------+---------------+-------------------+
|interaction_type|events|null_watch_time|non_null_watch_time|
+----------------+------+---------------+-------------------+
|        complete|  3726|           2300|               1426|
|            like|  3784|           3783|                  1|
|           pause|  3625|           3625|                  0|
|            play|  3592|           2201|               1391|
|           share|     1|              0|                  1|
+----------------+------+---------------+-------------------+



In [19]:
print("play / complete with NULL watch time:")

(
    df_kafka
    .filter(
        F.col("interaction_type").isin("play", "complete")
        & F.col("watch_time_sec").isNull()
    )
    .groupBy("interaction_type")
    .count()
    .show()
)

play / complete with NULL watch time:
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|        complete| 2300|
|            play| 2201|
+----------------+-----+



In [21]:
print("like / pause with non-NULL watch time:")

(
    df_kafka
    .filter(
        F.col("interaction_type").isin("like", "pause")
        & F.col("watch_time_sec").isNotNull()
    )
    .groupBy("interaction_type")
    .count()
    .show()
)

like / pause with non-NULL watch time:
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|            like|    1|
+----------------+-----+



In [22]:
print("=" * 70)
print("11. DEVICE TYPE DISTRIBUTION")
print("=" * 70)

(
    df_kafka
    .groupBy("device_type")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

11. DEVICE TYPE DISTRIBUTION
+-----------+-----+
|device_type|count|
+-----------+-----+
|tv         |5452 |
|web        |5352 |
|mobile     |5269 |
|console    |1    |
+-----------+-----+



In [23]:
print("=" * 70)
print("12. EVENT TIMESTAMP ANALYSIS")
print("=" * 70)

df_kafka.select(
    F.min("event_timestamp").alias("min_event_timestamp"),
    F.max("event_timestamp").alias("max_event_timestamp"),
).show()

12. EVENT TIMESTAMP ANALYSIS
+--------------------+-------------------+
| min_event_timestamp|max_event_timestamp|
+--------------------+-------------------+
|2032-04-22 16:57:...|    not-a-timestamp|
+--------------------+-------------------+



In [24]:
null_event_timestamp = (
    df_kafka
    .filter(F.col("event_timestamp").isNull())
    .count()
)

print(f"Null event_timestamp: {null_event_timestamp}")

Null event_timestamp: 0


In [25]:
print("=" * 70)
print("13. EVENTS OVER TIME")
print("=" * 70)

(
    df_kafka
    .withColumn("event_date", F.to_date("event_timestamp"))
    .groupBy("event_date")
    .count()
    .orderBy("event_date")
    .show(100, truncate=False)
)

13. EVENTS OVER TIME
+----------+-----+
|event_date|count|
+----------+-----+
|NULL      |1    |
|2032-04-22|384  |
|2032-04-23|1369 |
|2032-04-24|1390 |
|2032-04-25|1415 |
|2032-04-26|1367 |
|2032-04-27|1393 |
|2032-04-28|1396 |
|2032-04-29|1424 |
|2032-04-30|1385 |
|2032-05-01|1399 |
|2032-05-02|1399 |
|2032-05-03|866  |
|2032-05-04|102  |
|2032-05-05|56   |
|2032-05-09|112  |
|2032-05-11|224  |
|2032-05-12|168  |
|2032-05-13|168  |
|2032-05-14|56   |
+----------+-----+



In [26]:
(
    df_kafka
    .withColumn("event_hour", F.hour("event_timestamp"))
    .groupBy("event_hour")
    .count()
    .orderBy("event_hour")
    .show(24)
)

+----------+-----+
|event_hour|count|
+----------+-----+
|      NULL|    1|
|         0|  709|
|         1|  638|
|         2|  695|
|         3|  633|
|         4|  627|
|         5|  730|
|         6|  741|
|         7|  665|
|         8|  649|
|         9|  719|
|        10|  690|
|        11|  747|
|        12|  658|
|        13|  617|
|        14|  621|
|        15|  571|
|        16|  641|
|        17|  666|
|        18|  680|
|        19|  668|
|        20|  671|
|        21|  682|
|        22|  697|
+----------+-----+
only showing top 24 rows



In [27]:
print("=" * 70)
print("14. TIMESTAMP OBSERVATIONS")
print("=" * 70)

print("Earliest event:")
df_kafka.select(
    F.min("event_timestamp")
).show()

print("Latest event:")
df_kafka.select(
    F.max("event_timestamp")
).show()

14. TIMESTAMP OBSERVATIONS
Earliest event:
+--------------------+
|min(event_timestamp)|
+--------------------+
|2032-04-22 16:57:...|
+--------------------+

Latest event:
+--------------------+
|max(event_timestamp)|
+--------------------+
|     not-a-timestamp|
+--------------------+



In [28]:
print("=" * 70)
print("15. USER / VIDEO ID VALIDATION")
print("=" * 70)

print(
    "Null user_id:",
    df_kafka.filter(F.col("user_id").isNull()).count()
)

print(
    "Null video_id:",
    df_kafka.filter(F.col("video_id").isNull()).count()
)

15. USER / VIDEO ID VALIDATION
Null user_id: 0
Null video_id: 0


In [29]:
print("Empty user_id:")

df_kafka.filter(
    F.trim(F.col("user_id")) == ""
).count()

Empty user_id:


0

In [30]:
print("Empty video_id:")

df_kafka.filter(
    F.trim(F.col("video_id")) == ""
).count()

Empty video_id:


0

In [31]:
print("Empty video_id:")

df_kafka.filter(
    F.trim(F.col("video_id")) == ""
).count()

Empty video_id:


0

In [32]:
print("=" * 70)
print("16. DATA QUALITY OBSERVATIONS")
print("=" * 70)

print(
    "play/complete with NULL watch_time:",
    df_kafka.filter(
        F.col("interaction_type").isin("play", "complete")
        & F.col("watch_time_sec").isNull()
    ).count()
)

print(
    "like/pause with non-NULL watch_time:",
    df_kafka.filter(
        F.col("interaction_type").isin("like", "pause")
        & F.col("watch_time_sec").isNotNull()
    ).count()
)

print(
    "watch_time <= 0:",
    df_kafka.filter(
        F.col("watch_time_sec").isNotNull()
        & (F.col("watch_time_sec") <= 0)
    ).count()
)

print(
    "watch_time > 120:",
    df_kafka.filter(
        F.col("watch_time_sec").isNotNull()
        & (F.col("watch_time_sec") > 120)
    ).count()
)

print(
    "share interaction:",
    df_kafka.filter(
        F.col("interaction_type") == "share"
    ).count()
)

16. DATA QUALITY OBSERVATIONS
play/complete with NULL watch_time: 1
like/pause with non-NULL watch_time: 1
watch_time <= 0: 1
watch_time > 120: 4084
share interaction: 1


In [33]:
print("=" * 70)
print("KAFKA BRONZE — EDA SUMMARY")
print("=" * 70)

print(f"Total events: {df_kafka.count():,}")
print(f"Distinct users: {df_kafka.select('user_id').distinct().count():,}")
print(f"Distinct videos: {df_kafka.select('video_id').distinct().count():,}")
print(f"Distinct user-video pairs: {df_kafka.select('user_id', 'video_id').distinct().count():,}")

print(
    "Duplicate event_id groups:",
    df_kafka.groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nInteraction types:")
(
    df_kafka
    .groupBy("interaction_type")
    .count()
    .orderBy("interaction_type")
    .show(truncate=False)
)

print("\nDevice types:")
(
    df_kafka
    .groupBy("device_type")
    .count()
    .orderBy("device_type")
    .show(truncate=False)
)

print("\nTimestamp range:")
df_kafka.select(
    F.min("event_timestamp").alias("min"),
    F.max("event_timestamp").alias("max")
).show()

print("=" * 70)
print("EDA COMPLETE — NO TRANSFORMATIONS PERFORMED")
print("=" * 70)

KAFKA BRONZE — EDA SUMMARY
Total events: 16,074
Distinct users: 8,014
Distinct videos: 7,960
Distinct user-video pairs: 16,071
Duplicate event_id groups: 0

Interaction types:
+----------------+-----+
|interaction_type|count|
+----------------+-----+
|complete        |4138 |
|like            |4098 |
|pause           |3886 |
|play            |3951 |
|share           |1    |
+----------------+-----+


Device types:
+-----------+-----+
|device_type|count|
+-----------+-----+
|console    |1    |
|mobile     |5269 |
|tv         |5452 |
|web        |5352 |
+-----------+-----+


Timestamp range:
+--------------------+---------------+
|                 min|            max|
+--------------------+---------------+
|2032-04-22 16:57:...|not-a-timestamp|
+--------------------+---------------+

EDA COMPLETE — NO TRANSFORMATIONS PERFORMED
